In [2]:
import os
from typing import Dict, List
from dotenv import load_dotenv

load_dotenv()


class Config:

    # Supported platforms
    PLATFORMS = [
        "zomato",
        "swiggy",
        "doordash",
        "deliveroo",
        "instacart",
        "gorillas",
        "getir",
        "just_eat",
        "gopuff",
        "uber_eats"
    ]

    # Default settings
    DEFAULT_TIMEOUT = 30  # seconds
    DEFAULT_MAX_RETRIES = 3
    DEFAULT_RATE_LIMIT = 10  # requests per minute per platform

    @staticmethod
    def get_platform_api_key(platform: str) -> str:
        env_key = f"{platform.upper()}_API_KEY"
        return os.getenv(env_key, "")

    @staticmethod
    def get_all_api_keys() -> Dict[str, str]:
        keys = {}
        for platform in Config.PLATFORMS:
            key = Config.get_platform_api_key(platform)
            if key:
                keys[platform] = key
        return keys

    @staticmethod
    def get_rate_limit(platform: str) -> int:
        env_key = f"{platform.upper()}_RATE_LIMIT"
        return int(os.getenv(env_key, Config.DEFAULT_RATE_LIMIT))

    @staticmethod
    def get_timeout() -> int:
        return int(os.getenv("UQP_TIMEOUT", Config.DEFAULT_TIMEOUT))

    @staticmethod
    def get_max_retries() -> int:
        return int(os.getenv("UQP_MAX_RETRIES", Config.DEFAULT_MAX_RETRIES))


In [3]:
import hashlib
import hmac
import json
import os
import time
from datetime import datetime
from typing import Any, Dict

from dotenv import load_dotenv

try:
    from .config import Config
except ImportError:
    from config import Config


load_dotenv()


class UQPClient:

    RETRY_BACKOFF = 0.5

    def __init__(self):
        self.config = Config()

        self.requests_per_second = self.config.DEFAULT_RATE_LIMIT / 60
        self._rate_interval = 1.0 / self.requests_per_second

        self.max_retries = self.config.DEFAULT_MAX_RETRIES
        self.timeout = self.config.DEFAULT_TIMEOUT

        # API keys are loaded exclusively from environment variables.
        self._api_keys = {
            platform: os.getenv(f"{platform.upper()}_API_KEY")
            for platform in self.config.PLATFORMS
        }

        # Rate limiting is maintained independently for each platform.
        self._rate_limiter: Dict[str, float] = {}

        self._request_count: Dict[str, int] = {}

        self._endpoints = {
            "search": {
                "query": str,
                "location": str,
            },
            "get_price": {
                "item_id": str,
            },
            "get_offers": {},
            "create_order": {
                "items": list,
                "address": str,
                "payment_info": str,
            },
            "track_order": {
                "order_id": str,
            },
            "cancel_order": {
                "order_id": str,
            },
        }

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def send_request(
        self,
        platform: str,
        endpoint: str,
        params: Dict[str, Any],
    ) -> Dict[str, Any]:

        platform = self._normalize_platform(platform)
        endpoint = self._normalize_endpoint(endpoint)

        if not self._validate_request(platform, endpoint, params):
            raise ValueError("Invalid request")

        last_error = None

        for attempt in range(self.max_retries + 1):
            try:
                # Rate limiting is applied independently to each platform.
                self._wait_for_rate_limit(platform)

                signed_params = self._sign_request(
                    platform,
                    endpoint,
                    params,
                )

                response = self._simulate_api_call(
                    platform,
                    endpoint,
                    signed_params,
                )

                # Validate the simulated/platform response without adding
                # another helper function.
                if not isinstance(response, dict):
                    raise ValueError("Invalid response")

                required_response_fields = {
                    "platform",
                    "endpoint",
                    "data",
                    "timestamp",
                }

                if not required_response_fields.issubset(response.keys()):
                    raise ValueError("Invalid response")

                return self._normalize_response(
                    response,
                    platform,
                )

            except (TimeoutError, ConnectionError) as exc:
                last_error = exc

                if attempt >= self.max_retries:
                    break

                time.sleep(
                    self.RETRY_BACKOFF * (2 ** attempt)
                )

            except (ValueError, PermissionError):
                # Validation/authentication errors are not transient.
                raise

            except Exception as exc:
                last_error = exc

                if attempt >= self.max_retries:
                    break

                time.sleep(
                    self.RETRY_BACKOFF * (2 ** attempt)
                )

        if last_error is not None:
            return self._handle_uqp_error(
                last_error,
                platform,
            )

        return self._handle_uqp_error(
            RuntimeError("Request failed"),
            platform,
        )

    # ------------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------------

    def _validate_request(
        self,
        platform: str,
        endpoint: str,
        params: Dict[str, Any],
    ) -> bool:

        if not isinstance(platform, str):
            return False

        if not isinstance(endpoint, str):
            return False

        if not isinstance(params, dict):
            return False

        if platform not in self.SUPPORTED_PLATFORMS:
            return False

        if endpoint not in self._endpoints:
            return False

        endpoint_parameters = self._endpoints[endpoint]

        for parameter, expected_type in endpoint_parameters.items():
            if parameter not in params:
                return False

            value = params[parameter]

            if not isinstance(value, expected_type):
                return False

            if isinstance(value, str) and not value.strip():
                return False

        return True

    def _normalize_platform(self, platform: str) -> str:

        if not isinstance(platform, str):
            raise ValueError("Platform must be a string")

        platform = platform.strip().lower()

        if platform not in self.SUPPORTED_PLATFORMS:
            raise ValueError(
                f"Unsupported platform: {platform}"
            )

        return platform

    def _normalize_endpoint(self, endpoint: str) -> str:

        if not isinstance(endpoint, str):
            raise ValueError("Endpoint must be a string")

        endpoint = endpoint.strip().lower()

        if endpoint not in self._endpoints:
            raise ValueError(
                f"Unsupported endpoint: {endpoint}"
            )

        return endpoint

    # ------------------------------------------------------------------
    # Rate limiting
    # ------------------------------------------------------------------

    def _wait_for_rate_limit(self, platform: str) -> None:

        now = time.monotonic()

        last_request = self._rate_limiter.get(platform)

        if last_request is not None:
            elapsed = now - last_request

            if elapsed < self._rate_interval:
                time.sleep(
                    self._rate_interval - elapsed
                )

        self._rate_limiter[platform] = time.monotonic()

        self._request_count[platform] = (
            self._request_count.get(platform, 0) + 1
        )

    # ------------------------------------------------------------------
    # Authentication / signing
    # ------------------------------------------------------------------

    def _sign_request(
        self,
        platform: str,
        endpoint: str,
        params: Dict[str, Any],
    ) -> Dict[str, Any]:

        api_key = self._api_keys.get(platform)

        if not api_key:
            raise ValueError(
                f"No API key for platform: {platform}"
            )

        timestamp = str(int(time.time()))

        payload = json.dumps(
            {
                "endpoint": endpoint,
                "params": params,
                "timestamp": timestamp,
            },
            sort_keys=True,
            separators=(",", ":"),
        )

        signature = hmac.new(
            api_key.encode("utf-8"),
            payload.encode("utf-8"),
            hashlib.sha256,
        ).hexdigest()

        signed_params = dict(params)

        signed_params["_auth"] = {
            "timestamp": timestamp,
            "signature": signature,
        }

        return signed_params

    # ------------------------------------------------------------------
    # Simulated platform API
    # ------------------------------------------------------------------

    def _simulate_api_call(
        self,
        platform: str,
        endpoint: str,
        params: Dict[str, Any],
    ) -> Dict[str, Any]:

        api_key = self._api_keys.get(platform)

        if not api_key:
            raise PermissionError(
                f"No API key for platform: {platform}"
            )

        started = time.monotonic()

        # Simulated network delay.
        time.sleep(0.1)

        elapsed = time.monotonic() - started

        if elapsed > self.timeout:
            raise TimeoutError(
                f"Request to {platform} exceeded timeout"
            )

        return {
            "platform": platform,
            "endpoint": endpoint,
            "data": params,
            "timestamp": datetime.now().isoformat(),
        }

    # ------------------------------------------------------------------
    # Error handling
    # ------------------------------------------------------------------

    def _handle_uqp_error(
        self,
        error: Exception,
        platform: str,
    ) -> Dict[str, Any]:

        if isinstance(error, TimeoutError):
            code = "TIMEOUT"

        elif isinstance(error, PermissionError):
            code = "AUTHENTICATION_ERROR"

        elif isinstance(error, ConnectionError):
            code = "NETWORK_ERROR"

        elif isinstance(error, ValueError):
            code = "VALIDATION_ERROR"

        else:
            code = "PLATFORM_ERROR"

        return {
            "success": False,
            "error": {
                "code": code,
                "message": str(error),
                "platform": platform,
            },
            "timestamp": datetime.now().isoformat(),
        }

    # ------------------------------------------------------------------
    # Response normalization
    # ------------------------------------------------------------------

    def _normalize_response(
        self,
        response: Dict[str, Any],
        platform: str,
    ) -> Dict[str, Any]:

        if not isinstance(response, dict):
            raise ValueError("Response must be a dictionary")

        required_fields = {
            "platform",
            "endpoint",
            "data",
            "timestamp",
        }

        if not required_fields.issubset(response.keys()):
            raise ValueError("Invalid response format")

        data = response["data"]

        if not isinstance(data, dict):
            raise ValueError("Invalid response data")

        # Never expose authentication material to the caller.
        normalized_data = dict(data)
        normalized_data.pop("_auth", None)

        return {
            "success": True,
            "platform": platform,
            "endpoint": response["endpoint"],
            "data": normalized_data,
            "timestamp": response["timestamp"],
        }

In [ ]:
import logging
from typing import Dict, Any, List, Optional

try:
    from .uqp_client import UQPClient
except ImportError:
    from uqp_client import UQPClient

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Global UQP client instance
_uqp_client: Optional[UQPClient] = None


def _get_uqp_client() -> UQPClient:
    global _uqp_client
    if _uqp_client is None:
        _uqp_client = UQPClient()
    return _uqp_client


def search_items(
    query: str,
    location: str,
    user_id: str,
    platforms: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Search for food/items across quick commerce platforms.

    Args:
        query: Search query (item name, cuisine, etc.)
        location: Delivery location
        platforms: List of platforms to search (None = all platforms)

    Returns:
        Search results from all platforms
    """
    if not _check_authorization(user_id, "search_items"):
        raise ValueError("Unauthorized")

    if not _validate_tool_input(
        "search_items",
        query=query,
        location=location,
        platforms=platforms,
    ):
        raise ValueError("Invalid search_items input")

    # Do not log the location.
    logger.info("Searching for query=%s", query)

    client = _get_uqp_client()

    if platforms is None:
        platforms = [
            "zomato",
            "swiggy",
            "doordash",
            "deliveroo",
            "instacart",
        ]

    results = {}

    for platform in platforms:
        try:
            response = client.send_request(
                platform,
                "search",
                {
                    "query": query,
                    "location": location,
                },
            )

            results[platform] = response

        except Exception as exc:
            logger.error(
                "Search failed for platform=%s: %s",
                platform,
                exc,
            )

    return results


def get_prices(
    item_id: str,
    user_id: str,
    platforms: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Get prices for an item across platforms.

    Args:
        item_id: Item identifier
        platforms: List of platforms to check (None = all platforms)

    Returns:
        Prices from all platforms
    """
    if not _check_authorization(user_id, "get_prices"):
        raise ValueError("Unauthorized")

    if not _validate_tool_input(
        "get_prices",
        item_id=item_id,
        platforms=platforms,
    ):
        raise ValueError("Invalid get_prices input")

    client = _get_uqp_client()

    if platforms is None:
        platforms = [
            "zomato",
            "swiggy",
            "doordash",
            "deliveroo",
            "instacart",
        ]

    prices = {}

    for platform in platforms:
        try:
            response = client.send_request(
                platform,
                "get_price",
                {"item_id": item_id},
            )

            prices[platform] = response

        except Exception as exc:
            logger.error(
                "Price lookup failed for platform=%s: %s",
                platform,
                exc,
            )

    return prices


def get_offers(
    user_id: str,
    platforms: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Get current offers/discounts from each platform.

    Args:
        platforms: List of platforms to check (None = all platforms)

    Returns:
        Offers from all platforms
    """
    if not _check_authorization(user_id, "get_offers"):
        raise ValueError("Unauthorized")

    if not _validate_tool_input(
        "get_offers",
        platforms=platforms,
    ):
        raise ValueError("Invalid get_offers input")

    client = _get_uqp_client()

    if platforms is None:
        platforms = [
            "zomato",
            "swiggy",
            "doordash",
            "deliveroo",
            "instacart",
        ]

    offers = {}

    for platform in platforms:
        try:
            response = client.send_request(
                platform,
                "get_offers",
                {},
            )

            offers[platform] = response

        except Exception as exc:
            logger.error(
                "Offer lookup failed for platform=%s: %s",
                platform,
                exc,
            )

    return offers


def compare_options(
    item_id: str,
    user_id: str,
    platforms: Optional[List[str]] = None
) -> Dict[str, Any]:
    """
    Compare prices, offers, delivery times across platforms.

    Args:
        item_id: Item identifier
        platforms: List of platforms to compare (None = all platforms)

    Returns:
        Comparison results
    """
    if not _check_authorization(user_id, "compare_options"):
        raise ValueError("Unauthorized")

    if not _validate_tool_input(
        "compare_options",
        item_id=item_id,
        platforms=platforms
    ):
        raise ValueError("Invalid create_order input")

    prices = get_prices(item_id, platforms)
    offers_data = get_offers(platforms)

    comparison = {}

    for platform, price_response in prices.items():
        price_data = price_response.get("data", {})
        offer_response = offers_data.get(platform, {})
        offer_data = offer_response.get("data", {})

        comparison[platform] = {
            "price": price_data.get("price", 0),
            "offers": offer_data.get("offers", []),
        }

    return comparison

def create_order(
    platform: str,
    items: List[Dict[str, Any]],
    address: str,
    user_id: str,
    payment_info: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Create order on selected platform.

    Args:
        platform: Platform to order from
        items: List of items to order
        address: Delivery address
        payment_info: Payment information

    Returns:
        Order confirmation
    """
    if not _check_authorization(user_id, "create_order"):
        raise ValueError("Unauthorized")

    if not _validate_tool_input(
        "create_order",
        platform=platform,
        items=items,
        address=address,
        payment_info=payment_info
    ):
        raise ValueError("Invalid create_order input")

    client = _get_uqp_client()

    # Never log address or payment information.
    logger.info(
        "Creating order on platform=%s with %d item(s)",
        platform,
        len(items),
    )

    try:
        return client.send_request(
            platform,
            "create_order",
            {
                "items": items,
                "address": address,
                "payment_info": payment_info
            },
        )

    except Exception as exc:
        logger.error(
            "Order creation failed for platform=%s: %s",
            platform,
            exc,
        )
        raise


def track_order(
    platform: str,
    user_id: str,
    order_id: str
) -> Dict[str, Any]:
    """
    Track order delivery status.

    Args:
        platform: Platform name
        order_id: Order identifier

    Returns:
        Order tracking information
    """
    if not _check_authorization(user_id, "track_order"):
        raise ValueError("Unauthorized")

    if not _validate_tool_input(
        "track_order",
        platform=platform,
        order_id=order_id
    ):
        raise ValueError("Invalid track_order input")

    client = _get_uqp_client()

    try:
        return client.send_request(
            platform,
            "track_order",
            {"order_id": order_id},
        )

    except Exception as exc:
        logger.error(
            "Order tracking failed for platform=%s: %s",
            platform,
            exc,
        )
        raise


def cancel_order(
    platform: str,
    user_id: str,
    order_id: str
) -> Dict[str, Any]:
    """
    Cancel order with validation.

    Args:
        platform: Platform name
        order_id: Order identifier

    Returns:
        Cancellation confirmation
    """

    if not _validate_tool_input(
        "cancel_order",
        platform=platform,
        order_id=order_id
    ):
        raise ValueError("Invalid cancel_order input")

    client = _get_uqp_client()

    try:
        return client.send_request(
            platform,
            "cancel_order",
            {"order_id": order_id},
        )

    except Exception as exc:
        logger.error(
            "Order cancellation failed for platform=%s: %s",
            platform,
            exc,
        )
        raise

def register_tools() -> List[Dict[str, Any]]:
    """
    Register MCP tools with proper schemas.
    """

    return [
        {
            "name": "search_items",
            "description": (
                "Search for food and other items across "
                "quick commerce platforms."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Item or food to search for.",
                    },
                    "location": {
                        "type": "string",
                        "description": "Delivery location.",
                    },
                    "platforms": {
                        "type": "array",
                        "items": {
                            "type": "string",
                            "enum": [
                                "zomato",
                                "swiggy",
                                "doordash",
                                "deliveroo",
                                "instacart",
                            ],
                        },
                        "description": (
                            "Platforms to search. "
                            "Omit to search all supported platforms."
                        ),
                    },
                },
                "required": ["query", "location"],
                "additionalProperties": False,
            },
        },
        {
            "name": "get_prices",
            "description": (
                "Get the price of an item across "
                "quick commerce platforms."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "item_id": {
                        "type": "string",
                        "description": "Item identifier.",
                    },
                    "platforms": {
                        "type": "array",
                        "items": {
                            "type": "string",
                            "enum": [
                                "zomato",
                                "swiggy",
                                "doordash",
                                "deliveroo",
                                "instacart",
                            ],
                        },
                    },
                },
                "required": ["item_id"],
                "additionalProperties": False,
            },
        },
        {
            "name": "get_offers",
            "description": (
                "Get current offers and discounts "
                "from quick commerce platforms."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "platforms": {
                        "type": "array",
                        "items": {
                            "type": "string",
                            "enum": [
                                "zomato",
                                "swiggy",
                                "doordash",
                                "deliveroo",
                                "instacart",
                            ],
                        },
                    },
                },
                "additionalProperties": False,
            },
        },
        {
            "name": "compare_options",
            "description": (
                "Compare item prices and available offers "
                "across quick commerce platforms."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "item_id": {
                        "type": "string",
                        "description": "Item identifier.",
                    },
                    "platforms": {
                        "type": "array",
                        "items": {
                            "type": "string",
                            "enum": [
                                "zomato",
                                "swiggy",
                                "doordash",
                                "deliveroo",
                                "instacart",
                            ],
                        },
                    },
                },
                "required": ["item_id"],
                "additionalProperties": False,
            },
        },
        {
            "name": "create_order",
            "description": (
                "Create an order on a selected quick commerce platform."
            ),
            "input_schema": {
                "type": "object",
                "properties": {
                    "platform": {
                        "type": "string",
                        "enum": [
                            "zomato",
                            "swiggy",
                            "doordash",
                            "deliveroo",
                            "instacart",
                        ],
                    },
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                        },
                        "minItems": 1,
                    },
                    "address": {
                        "type": "string",
                        "description": "Delivery address.",
                    },
                    "payment_info": {
                        "type": "object",
                        "description": "Payment information.",
                    },
                },
                "required": [
                    "platform",
                    "items",
                    "address",
                    "payment_info",
                ],
                "additionalProperties": False,
            },
        },
        {
            "name": "track_order",
            "description": "Track the delivery status of an order.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "platform": {
                        "type": "string",
                        "enum": [
                            "zomato",
                            "swiggy",
                            "doordash",
                            "deliveroo",
                            "instacart",
                        ],
                    },
                    "order_id": {
                        "type": "string",
                        "description": "Order identifier.",
                    },
                },
                "required": ["platform", "order_id"],
                "additionalProperties": False,
            },
        },
        {
            "name": "cancel_order",
            "description": "Cancel an existing order.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "platform": {
                        "type": "string",
                        "enum": [
                            "zomato",
                            "swiggy",
                            "doordash",
                            "deliveroo",
                            "instacart",
                        ],
                    },
                    "order_id": {
                        "type": "string",
                        "description": "Order identifier.",
                    },
                },
                "required": ["platform", "order_id"],
                "additionalProperties": False,
            },
        },
    ]


def _validate_tool_input(tool_name: str, **kwargs) -> bool:
    """
    Validate all tool inputs.

    Args:
        tool_name: Name of tool
        **kwargs: Tool parameters

    Returns:
        True if valid, False otherwise
    """
    if not isinstance(tool_name, str) or not tool_name.strip():
        return False

    tools = register_tools()

    tool = next(
        (
            tool
            for tool in tools
            if tool.get("name") == tool_name.strip()
        ),
        None,
    )

    if tool is None:
        return False

    schema = tool.get("input_schema")

    if not isinstance(schema, dict):
        return False

    properties = schema.get("properties", {})
    required = schema.get("required", [])
    additional_properties = schema.get(
        "additionalProperties",
        True,
    )

    if not isinstance(properties, dict):
        return False

    if not isinstance(required, list):
        return False

    # ------------------------------------------------------------------
    # Required parameters
    # ------------------------------------------------------------------

    for parameter in required:
        if parameter not in kwargs:
            return False

    # ------------------------------------------------------------------
    # Unkwnown parameters
    # ------------------------------------------------------------------

    if additional_properties is False:
        for parameter in kwargs:
            if parameter not in properties:
                return False

    # ------------------------------------------------------------------
    # Validate supplied parameters
    # ------------------------------------------------------------------

    for parameter, value in kwargs.items():

        parameter_schema = properties.get(parameter)

        if parameter_schema is None:
            if additional_properties is False:
                return False

            continue

        if value is None:
            if parameter not in required:
                continue

            return False

        if not validate_value(value, parameter_schema):
            return False

    return True

def validate_value(value: Any, value_schema: Dict[str, Any]) -> bool:
    """
    Recursively validate a value against a JSON-schema-like definition.
    """

    if not isinstance(value_schema, dict):
        return False

    # --------------------------------------------------------------
    # Enum
    # --------------------------------------------------------------
    if "enum" in value_schema:
        enum_values = value_schema["enum"]

        if not isinstance(enum_values, list):
            return False
        if value not in enum_values:
            return False

    # --------------------------------------------------------------
    # Type
    # --------------------------------------------------------------

    expected_type = value_schema.get("type")

    if expected_type == "string":

        if not isinstance(value, str):
            return False

        if not value.strip():
            return False

    elif expected_type == "number":

        if isinstance(value, bool):
            return False

        if not isinstance(value, (int, float)):
            return False

    elif expected_type == "integer":

        if isinstance(value, bool):
            return False

        if not isinstance(value, int):
            return False

    elif expected_type == "boolean":

        if not isinstance(value, bool):
            return False

    elif expected_type == "array":

        if not isinstance(value, list):
            return False

        item_schema = value_schema.get("items")

        if item_schema is not None:
            for item in value:
                if not validate_value(item, item_schema):
                    return False

        min_items = value_schema.get("minItems")

        if min_items is not None and len(value) < min_items:
            return False

        max_items = value_schema.get("maxItems")

        if max_items is not None and len(value) > max_items:
            return False

    elif expected_type == "object":

        if not isinstance(value, dict):
            return False

        object_properties = value_schema.get(
            "properties",
            {},
        )

        object_required = value_schema.get(
            "required",
            [],
        )

        object_additional = value_schema.get(
            "additionalProperties",
            True,
        )

        if not isinstance(object_properties, dict):
            return False

        if not isinstance(object_required, list):
            return False

        # Required nested properties
        for parameter in object_required:
            if parameter not in value:
                return False

        # Unknown nested properties
        if object_additional is False:
            for parameter in value:
                if parameter not in object_properties:
                    return False

        # Recursive nested validation
        for parameter, parameter_value in value.items():

            parameter_schema = object_properties.get(parameter)

            if parameter_schema is None:
                continue

            if not validate_value(
                parameter_value,
                parameter_schema,
            ):
                return False

    else:
        return False

    return True


def _check_authorization(user_id: str, tool_name: str) -> bool:
    """
    Verify user has permission to call tool.

    Args:
        user_id: User identifier
        tool_name: Tool name

    Returns:
        True if authorized, False otherwise
    """

    if not isinstance(user_id, str) or not user_id.strip():
        return False

    if not isinstance(tool_name, str) or not tool_name.strip():
        return False

    allowed_tools = {
        "search_items",
        "get_prices",
        "get_offers",
        "compare_options",
        "create_order",
        "track_order",
        "cancel_order",
    }

    return tool_name in allowed_tools


def _sanitize_output(data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Remove sensitive data from output before logging.

    Args:
        data: Data to sanitize

    Returns:
        Sanitized data
    """
    if not isinstance(data, dict):
        raise ValueError("Data must be a dictionary")

    sensitive_keys = {
        "payment_info",
        "payment",
        "card_number",
        "cardNumber",
        "cvv",
        "cvc",
        "security_code",
        "api_key",
        "apiKey",
        "secret",
        "token",
        "authorization",
        "address",
        "delivery_address",
    }

    sanitized = {}

    for key, value in data.items():
        if key in sensitive_keys:
            sanitized[key] = "[REDACTED]"
            continue

        if isinstance(value, dict):
            sanitized[key] = _sanitize_output(value)
        elif isinstance(value, list):
            sanitized[key] = [
                _sanitize_output(item)
                if isinstance(item, dict)
                else item
                for item in value
            ]
        else:
            sanitized[key] = value

    return sanitized

In [ ]:
import math
from bisect import insort_right
from collections import Counter, defaultdict
from datetime import timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Set

try:
    from llm import LLMClient
    client = LLMClient().client()
except Exception:
    client = None


def cosine_similarity(v1: List[float], v2: List[float]) -> float:
    dot_product = sum(a * b for a, b in zip(v1, v2))
    norm_v1 = math.sqrt(sum(a * a for a in v1))
    norm_v2 = math.sqrt(sum(b * b for b in v2))
    if not norm_v1 or not norm_v2:
        return 0.0
    return dot_product / (norm_v1 * norm_v2)


class LightweightIndexer:

    def __init__(self):
        self.by_platform: Dict[str, Set[str]] = defaultdict(set)
        self.by_item: Dict[str, Set[str]] = defaultdict(set)
        self.timeline: List[tuple[str, str]] = []


    def index_order(self, order_id: str, order: Dict[str, Any]) -> None:
        platform = order.get("platform", "")
        if platform:
            self.by_platform[platform.lower()].add(order_id)

        items = order.get("items", [])
        for item in items:
            item_name = item.get("name") if isinstance(item, dict) else str(item)
            if item_name:
                self.by_item[item_name.lower()].add(order_id)

        timestamp = order.get("timestamp", datetime.now().isoformat())
        insort_right(self.timeline, (timestamp, order_id))


    def remove_order(self, order_id: str, order: Dict[str, Any]) -> None:
        platform = order.get("platform")
        if platform and platform.lower() in self.by_platform:
            self.by_platform[platform.lower()].discard(order_id)

        items = order.get("items", [])
        for item in items:
            item_name = item.get("name") if isinstance(item, dict) else str(item)
            if item_name and item_name.lower() in self.by_item:
                self.by_item[item_name.lower()].discard(order_id)

        self.timeline = [entry for entry in self.timeline if entry[1] != order_id]


class MemoryStore:

    def __init__(self, user_id: str, storage_path: Optional[str] = None):
        self.user_id = user_id
        self.storage_path = Path(storage_path) if storage_path else None
        self.preferences: Dict[str, Any] = {}
        # Dict of order_id -> order_data for O(1) lookups
        self.order_history: Dict[str, Dict[str, Any]] = {}
        self.favorite_items: List[Dict[str, Any]] = []
        self.indexer = LightweightIndexer()

        # Load existing data if file exists
        if self.storage_path and self.storage_path.exists():
            self._load_from_storage()


    def _load_from_storage(self) -> None:
        with open(self.storage_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        self.preferences = data.get("preferences", {})
        self.favorite_items = data.get("favorite_items", [])

        # Handle both list and dict formats gracefully on load
        raw_orders = data.get("order_history", {})
        if isinstance(raw_orders, list):
            self.order_history = {
                str(order.get("order_id", order.get("id", idx))): order
                for idx, order in enumerate(raw_orders)
            }
        else:
            self.order_history = raw_orders

        # Rebuild indexes
        self.indexer = LightweightIndexer()
        for order_id, order in self.order_history.items():
            self.indexer.index_order(order_id, order)


    def _save_to_storage(self) -> None:
        if not self.storage_path:
            return

        self.storage_path.parent.mkdir(parents=True, exist_ok=True)
        data = {
            "user_id": self.user_id,
            "preferences": self.preferences,
            "order_history": self.order_history,
            "favorite_items": self.favorite_items,
            "last_updated": datetime.now().isoformat(),
        }

        temp_file = self.storage_path.with_suffix(".tmp")
        with open(temp_file, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        os.replace(temp_file, self.storage_path)


    def store_preference(self, preference_type: str, value: Any) -> None:
        if not preference_type or not isinstance(preference_type, str):
            raise ValueError("preference_type must be a non-empty string")

        self.preferences[preference_type.strip().lower()] = value
        self._save_to_storage()


    def store_order_history(self, order: Dict[str, Any]) -> str:
        if not isinstance(order, dict):
            raise ValueError("Order must be a dictionary")

        order_record = order.copy()
        order_id = str(
            order_record.get(
                "order_id",
                order_record.get("id", f"ord_{len(self.order_history) + 1}_{int(datetime.now().timestamp())}"),
            )
        )
        order_record["order_id"] = order_id
        order_record["user_id"] = self.user_id

        if "timestamp" not in order_record:
            order_record["timestamp"] = datetime.now().isoformat()

        # Save to primary storage
        self.order_history[order_id] = order_record

        # Update lightweight indexes
        self.indexer.index_order(order_id, order_record)

        # Update preference learning automatically
        self._auto_learn_preferences()

        self._save_to_storage()
        return order_id


    def store_favorite_items(self, items: List[Dict[str, Any]]) -> None:
        if not isinstance(items, list):
            raise ValueError("Items must be a list")
        self.favorite_items.extend(items)
        self._save_to_storage()


    def retrieve_preferences(self, semantic_query: Optional[str] = None, preference_type: Optional[str] = None) -> Dict[str, Any]:
        if preference_type:
            key = preference_type.strip().lower()
            return {key: self.preferences.get(key)}

        if semantic_query:
            return {k: v for k, v in self.preferences.items() if semantic_query in k}

        return self.preferences


    def get_order_history(self, platform: Optional[str] = None, limit: int = 10) -> List[Dict[str, Any]]:
        if platform:
            candidate_ids = self.indexer.by_platform.get(platform.lower(), set())
            orders = [self.order_history[oid] for oid in candidate_ids if oid in self.order_history]
        else:
            orders = list(self.order_history.values())

        orders.sort(key=lambda x: x.get("timestamp", ""), reverse=True)
        return orders[:limit]


    def get_favorite_items(self) -> List[Dict[str, Any]]:
        return self.favorite_items


    def search_similar_orders(self, query: Dict[str, Any], top_k: int = 3) -> List[Dict[str, Any]]:
        if not client:
            return []

        try:
            # Format query dictionary as readable text for embedding
            query_str = json.dumps(query) if isinstance(query, dict) else str(query)
            response = client.embeddings.create(
                model="text-embedding-3-small",
                input=query_str,
            )
            query_vector = response.data[0].embedding
        except Exception as e:
            print(f"Error creating query embedding: {e}")
            return []

        scored_orders = []
        for order in self.order_history.values():
            try:
                order_summary = f"Platform: {order.get('platform', '')}. Items: {json.dumps(order.get('items', []))}"
                order_resp = client.embeddings.create(
                    model="text-embedding-3-small",
                    input=order_summary,
                )
                order_vector = order_resp.data[0].embedding
                score = cosine_similarity(query_vector, order_vector)
                scored_orders.append((score, order))
            except Exception:
                continue

        scored_orders.sort(key=lambda x: x[0], reverse=True)
        return [order for _, order in scored_orders[:top_k]]


    def _auto_learn_preferences(self) -> None:
        platform_counts = Counter(
            order.get("platform") for order in self.order_history.values() if order.get("platform")
        )
        if platform_counts:
            most_frequent_platform = platform_counts.most_common(1)[0][0]
            self.preferences["preferred_platform"] = most_frequent_platform


    def get_preferred_platform(self) -> Optional[str]:
        return self.preferences.get("preferred_platform")


    def cleanup_old_data(self, days_to_keep: int = 90) -> None:
        cutoff_date = datetime.now() - timedelta(days=days_to_keep)

        orders_to_delete = []
        for order_id, order in self.order_history.items():
            ts_str = order.get("timestamp")
            if ts_str:
                try:
                    order_dt = datetime.fromisoformat(ts_str)
                    if order_dt < cutoff_date:
                        orders_to_delete.append((order_id, order))
                except ValueError:
                    pass

        for order_id, order in orders_to_delete:
            del self.order_history[order_id]
            self.indexer.remove_order(order_id, order)

        if orders_to_delete:
            self._save_to_storage()


In [ ]:
from typing import Dict, Any, List, Optional


class PriceComparator:

    def __init__(self, user_preferences: Optional[Dict[str, Any]] = None):
        """
        Initialize price comparator.

        Args:
            user_preferences: User preferences (prefer_fast_delivery, prefer_cheap, etc.)
        """
        self.user_preferences = user_preferences or {}


    def compare_prices(self, prices: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
        """
        Compare prices across platforms.

        Args:
            prices: Dictionary of platform -> price data

        Returns:
            Comparison results
        """
        if not isinstance(prices, dict):
            raise ValueError("prices must be a dictionary")

        comparison = {}

        for platform, price_data in prices.items():
            if not isinstance(price_data, dict):
                continue

            base_price = float(price_data.get("price", 0) or 0)
            delivery_charge = float(price_data.get("delivery_charge", 0) or 0)
            tax = float(price_data.get("tax", 0) or 0)

            if base_price < 0 or delivery_charge < 0 or tax < 0:
                continue

            comparison[platform] = {
                "base_price": base_price,
                "delivery_charge": delivery_charge,
                "tax": tax,
                "total": self._normalize_price(price_data, platform),
            }

        return comparison


    def compare_offers(self, offers: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
        """
        Compare offers/discounts and calculate final price.

        Args:
            offers: Dictionary of platform -> offers data

        Returns:
            Comparison with final prices after offers
        """
        if not isinstance(offers, dict):
            raise ValueError("offers must be a dictionary")

        comparison = {}
        for platform, offer_data in offers.items():
            if not isinstance(offer_data, dict):
                continue

            offers_list = offer_data.get("offers", [])

            if not isinstance(offers_list, list):
                offers_list = []

            base_price = float(
                offer_data.get(
                    "price",
                    offer_data.get("base_price", 0),
                ) or 0
            )

            best_offer = None
            best_price = base_price

            for offer in offers_list:
                if not isinstance(offer, dict):
                    continue

                try:
                    value = float(offer.get("value", 0) or 0)
                    min_order = float(offer.get("min_order", 0) or 0)
                except (ValueError, TypeError):
                    continue

                if value < 0 or base_price < min_order:
                    continue

                offer_type = offer.get("type", "").lower()

                if offer_type == "percentage":
                    # Percentage discount
                    discounted_price = base_price * (value / 100.0)
                elif offer_type in ("fixed", "amount", "flat"):
                    discounted_price = value
                else:
                    continue

                discount = min(discounted_price, base_price)
                final_price = max(0.0, base_price - discount)

                if final_price < best_price:
                    best_price = final_price
                    best_offer = offer

            comparison[platform] = {
                "offers": offers_list,
                "best_offer": best_offer,
                "discount": round(base_price - best_price, 2),
                "final_price": round(best_price, 2)
            }

        return comparison


    def compare_delivery_times(self, delivery_times: Dict[str, int]) -> Dict[str, Any]:
        """
        Compare estimated delivery times.

        Args:
            delivery_times: Dictionary of platform -> delivery time in minutes

        Returns:
            Comparison results
        """
        if not isinstance(delivery_times, dict):
            raise ValueError("delivery_times must be a dictionary")

        comparison = {}
        for platform, time_minutes in delivery_times.items():
            try:
                time_minutes = int(time_minutes)
            except (ValueError, TypeError):
                continue

            if time_minutes < 0:
                continue

            comparison[platform] = {
                "delivery_time_minutes": time_minutes,
                "is_fast": time_minutes < 30  # Fast if under 30 minutes
            }

        return comparison


    def rank_options(self, options: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Rank options based on price, offers, delivery time, user preferences.

        Args:
            options: List of options with price, offers, delivery_time

        Returns:
            Ranked list of options (best first)

        BUGS:
        - No ranking algorithm
        - Doesn't factor in user preferences
        """
        if not isinstance(options, list):
            raise ValueError("options must be a list")

        valid_options = [
            option
            for option in options
            if isinstance(option, dict)
        ]

        if not valid_options:
            return []

        prefer_fast = bool(
            self.user_preferences.get("fast_delivery", False)
        )

        prefer_cheap = bool(
            self.user_preferences.get("cheap_price", False)
        )

        preferred_platforms = self.user_preferences.get("preferred_platforms", [])

        def get_price(option: Dict[str, Any]) -> float:
            return float(
                option.get(
                    "final_price",
                    option.get(
                        "total_price",
                        option.get("total", float('inf'))
                    )
                )
                or 0
            )

        def get_discount(option: Dict[str, Any]) -> float:
            return float(
                option.get("discount", 0) or 0
            )

        def get_delivery_time(option: Dict[str, Any]) -> float:
            return float(
                option.get("delivery_time_minutes", float("inf"))
            )

        def ranking_key(option: Dict[str, Any]):
            platform = option.get("platform")

            price = get_price(option)
            discount = get_discount(option)
            delivery_time = get_delivery_time(option)

            if prefer_fast:
                primary = delivery_time
                secondary = price
            elif prefer_cheap:
                primary = price
                secondary = delivery_time
            else:
                primary = price
                secondary = delivery_time

            platform_priority = (
                0 if platform in preferred_platforms else 1
            )

            return (
                platform_priority,
                primary,
                secondary,
                -discount
            )

        return sorted(valid_options, key=ranking_key)


    def get_best_option(self, comparison_data: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        """
        Determine best option considering all factors.

        Args:
            comparison_data: Combined comparison data (prices, offers, delivery times)

        Returns:
            Best option or None
        """
        if not comparison_data or not isinstance(comparison_data, dict):
            return None

        options = []

        for platform, data in comparison_data.items():
            if not isinstance(data, dict):
                continue

            option = {
                "platform": platform,
                **data
            }
            options.append(option)

        if not options:
            return None

        ranked_options = self.rank_options(options)
        return ranked_options[0] if ranked_options else None

    def _normalize_price(self, price_data: Dict[str, Any], platform: str) -> float:
        """
        Normalize price (add taxes, delivery charges) for fair comparison.

        Args:
            price_data: Price data from platform
            platform: Platform name

        Returns:
            Normalized total price
        """
        if not isinstance(price_data, dict):
            raise ValueError("Invalid price data")

        try:
            current_price = float(price_data.get("price", 0) or 0)
            tax = float(price_data.get("tax", 0) or 0)
            delivery_charge = float(price_data.get("delivery_charge", 0) or 0)
        except (TypeError, ValueError):
            raise ValueError("Invalid price data")

        if current_price < 0 or tax < 0 or delivery_charge < 0:
            raise ValueError("Invalid price data")

        return round(
            current_price + tax + delivery_charge,
            2
        )


    def _calculate_final_price(self, base_price: float, offers: List[Dict[str, Any]]) -> float:
        """
        Calculate final price after applying offers/discounts.

        Args:
            base_price: Base price
            offers: List of available offers

        Returns:
            Final price after best offer
        """
        try:
            best_price = float(base_price)
        except (TypeError, ValueError):
            raise ValueError("Invalid base price")

        if base_price < 0:
            raise ValueError("Invalid base price")

        if not isinstance(offers, list):
            raise ValueError("Invalid offers list")

        best_price = base_price

        for offer in offers:
            if not isinstance(offer, dict):
                continue

            try:
                value = float(offer.get("value", 0) or 0)
                min_order = float(offer.get("min_order", 0) or 0)
            except (TypeError, ValueError):
                continue

            if value < 0 or base_price < min_order:
                continue

            offer_type = offer.get("type")

            if offer_type == "percentage":
                discount = base_price * (value / 100.0)
            elif offer_type in ("fixed", "amount", "flat"):
                discount = value
            else:
                continue

            discount = min(discount, base_price)
            final_price = max(0.0, base_price - discount)

            if final_price < best_price:
                best_price = final_price

        return round(best_price, 2)

In [ ]:
import asyncio
import time
from typing import Dict, Any, Optional, List
from datetime import datetime, timedelta
from enum import Enum

# BUG: No approval queue - approvals stored in memory only
_approval_queue: Dict[str, Dict[str, Any]] = {}


class ApprovalStatus(Enum):
    """Approval status"""
    PENDING = "pending"
    APPROVED = "approved"
    REJECTED = "rejected"
    EXPIRED = "expired"


class HITLWorkflow:
    """Human-in-the-loop approval workflow manager"""

    def __init__(self, default_timeout: int = 300):
        """
        Initialize HITL workflow.

        Args:
            default_timeout: Default approval timeout in seconds
        """
        self.default_timeout = default_timeout
        self.approval_history: List[Dict[str, Any]] = []

    def request_order_approval(self, order_summary: Dict[str, Any], user_id: str, comparison_data: Optional[Dict[str, Any]] = None) -> str:
        """
        Create approval request with order summary and price comparison.

        Args:
            order_summary: Order details (platform, items, price, delivery_time)
            comparison_data: Optional price comparison data

        Returns:
            Approval request ID

        BUGS:
        - Doesn't show order summary in approval request
        - Doesn't integrate price comparison
        """
        if not isinstance(order_summary, dict) or not order_summary:
            raise ValueError("Invalid order summary")

        if comparison_data is not None and not isinstance(comparison_data, dict):
            raise ValueError("comparison_data must be a dictionary")

        approval_id = f"approval_{int(time.time())}"

        now = datetime.now()
        expired_at = now + timedelta(seconds=self.default_timeout)

        formatted_request = self._format_approval_request(order_summary, comparison_data)

        approval_request = {
            "approval_id": approval_id,
            "status": ApprovalStatus.PENDING.value,
            "user_id": user_id,
            "order_summary": order_summary,
            "comparison_data": comparison_data,
            "formatted_request": formatted_request,
            "created_at": now.isoformat(),
            "expires_at": expired_at.isoformat()
        }

        _approval_queue[approval_id] = approval_request

        self.approval_history.append({
            "approval_id": approval_id,
            "event": "created",
            "status": ApprovalStatus.PENDING.value,
            "timestamp": now.isoformat()
        })

        self._notify_user(
            approval_id,
            order_summary
        )

        return approval_id

    async def wait_for_approval(self, approval_id: str, timeout: Optional[int] = None) -> Dict[str, Any]:
        """
        Wait for approval with timeout.

        Args:
            approval_id: Approval request ID
            timeout: Optional timeout in seconds (default: self.default_timeout)

        Returns:
            Approval result (status, reason)

        BUGS:
        - Missing timeout handling
        - No approval expiration check
        """
        if (
            not isinstance(approval_id, str) or
            not approval_id.strip()
        ):
            return {
                "status": "error",
                "reason": "Invalid approval ID",
                "approval_id": approval_id,
            }

        approval = _approval_queue.get(approval_id)

        if approval is None:
            return {"status": "not_found", "approval_id": approval_id}

        if timeout is None:
            timeout = self.default_timeout

        if not isinstance(timeout, int) or timeout <= 0:
            return {"status": "error", "reason": "Invalid timeout", "approval_id": approval_id}

        try:
            expires_at = datetime.fromisoformat( approval["expires_at"] )
        except (KeyError, ValueError, TypeError):
            return {
                "status": "error",
                "reason": "Invalid approval expiration",
                "approval_id": approval_id,
            }

        async def poll():
            while True:
                approval = _approval_queue.get(approval_id)
                if approval is None:
                    return {"status": "not_found", "approval_id": approval_id}

                status = approval.get("status")
                if status == ApprovalStatus.APPROVED.value:
                    return {"status": "approved", "approval_id": approval_id}

                if status == ApprovalStatus.REJECTED.value:
                    return {"status": "rejected", "reason": approval.get("rejection_reason"), "approval_id": approval_id}

                if status == ApprovalStatus.EXPIRED.value:
                    return {"status": "expired", "approval_id": approval_id}

                now = datetime.now()

                if now >= expires_at:
                    approval["status"] = ApprovalStatus.EXPIRED.value
                    approval["expired_at"] = now.isoformat()
                    self.approval_history.append({
                        "approval_id": approval_id,
                        "event": "expired",
                        "status": ApprovalStatus.EXPIRED.value,
                        "timestamp": now.isoformat(),
                    })
                    return {"status": "expired", "approval_id": approval_id}

                await asyncio.sleep(0.5)

        try:
            return await asyncio.wait_for(poll(), timeout=timeout)
        except asyncio.TimeoutError:
            return {"status": "timeout", "approval_id": approval_id}

    def approve_order(self, approval_id: str, user_id: Optional[str] = None) -> bool:
        """
        Process approval with validation.

        Args:
            approval_id: Approval request ID
            user_id: Optional user ID for validation

        Returns:
            True if approved, False otherwise
        """
        if (
            not isinstance(approval_id, str) or
            not approval_id.strip()
        ):
            return False

        approval = _approval_queue.get(approval_id)
        if approval is None:
            return False

        if approval.get("status") != ApprovalStatus.PENDING.value:
            return False

        try:
            expires_at = datetime.fromisoformat(approval["expires_at"])
        except (KeyError, ValueError, TypeError):
            return False

        now = datetime.now()
        if now >= expires_at:
            approval["status"] = ApprovalStatus.EXPIRED.value
            approval["expired_at"] = now.isoformat()
            self.approval_history.append({
                "approval_id": approval_id,
                "event": "expired",
                "status": ApprovalStatus.EXPIRED.value,
                "timestamp": now.isoformat(),
            })
            return False

        approval["status"] = ApprovalStatus.APPROVED.value
        approval["approved_at"] = now.isoformat()
        approval["approved_by"] = user_id

        self.approval_history.append({
            "approval_id": approval_id,
            "event": "approved",
            "status": ApprovalStatus.APPROVED.value,
            "user_id": user_id,
            "timestamp": now.isoformat(),
        })

        return True

    def reject_order(self, approval_id: str, reason: str, user_id: Optional[str] = None) -> bool:
        """
        Handle rejection with reason.

        Args:
            approval_id: Approval request ID
            reason: Rejection reason
            user_id: Optional user ID

        Returns:
            True if rejected, False otherwise
        """
        if (
            not isinstance(approval_id, str)
            or not approval_id.strip()
        ):
            return False

        if not isinstance(reason, str) or not reason.strip():
            return False

        approval = _approval_queue.get(approval_id)
        if approval is None:
            return False

        if approval.get("status") != ApprovalStatus.PENDING.value:
            return False

        try:
            expires_at = datetime.fromisoformat(approval["expires_at"])
        except (KeyError, ValueError, TypeError):
            return False

        now = datetime.now()
        if now >= expires_at:
            approval["status"] = ApprovalStatus.EXPIRED.value
            approval["expired_at"] = now.isoformat()
            self.approval_history.append({
                "approval_id": approval_id,
                "event": "expired",
                "status": ApprovalStatus.EXPIRED.value,
                "timestamp": now.isoformat(),
            })
            return False

        approval["status"] = ApprovalStatus.REJECTED.value
        approval["rejection_reason"] = reason.strip()
        approval["rejected_at"] = now.isoformat()
        approval["rejected_by"] = user_id

        self.approval_history.append({
            "approval_id": approval_id,
            "event": "rejected",
            "status": ApprovalStatus.REJECTED.value,
            "user_id": user_id,
            "reason": reason.strip(),
            "timestamp": now.isoformat(),
        })

        return True

    def get_pending_approvals(self, user_id: Optional[str] = None) -> List[Dict[str, Any]]:
        """
        List pending approvals.

        Args:
            user_id: Optional filter by user

        Returns:
            List of pending approvals
        """
        pending = []
        now = datetime.now()

        for approval_id, approval in _approval_queue.items():

            if approval.get("status") != ApprovalStatus.PENDING.value:
                continue

            try:
                expires_at = datetime.fromisoformat(approval["expires_at"])
            except (KeyError, ValueError, TypeError):
                continue

            if now >= expires_at:
                approval["status"] = ApprovalStatus.EXPIRED.value
                approval["expired_at"] = now.isoformat()
                self.approval_history.append({
                    "approval_id": approval_id,
                    "event": "expired",
                    "status": ApprovalStatus.EXPIRED.value,
                    "timestamp": now.isoformat(),
                })
                continue

            if user_id is not None:
                if approval.get("user_id") != user_id:
                    continue

            pending.append(dict(approval))

        return pending

    def _notify_user(self, approval_id: str, order_summary: Dict[str, Any]) -> None:
        """
        Notify user of pending approval.

        Args:
            approval_id: Approval request ID
            order_summary: Order summary to show in notification
        """
        if (
            not approval_id or
            not isinstance(order_summary, dict)
        ):
            return

        return None

    def _format_approval_request(self, order_summary: Dict[str, Any], comparison_data: Optional[Dict[str, Any]]) -> str:
        """
        Format approval request with comparison data.

        Args:
            order_summary: Order summary
            comparison_data: Price comparison data

        Returns:
            Formatted approval request text
        """
        if not isinstance(order_summary, dict):
            raise ValueError("order_summary must be a dictionary")

        lines = [
            "Order approval required",
            "",
        ]

        platform = order_summary.get("platform")

        if platform:
            lines.append(f"Platform: {platform}")

        items = order_summary.get("items", [])

        if items:
            lines.append("Items:")
            for item in items:
                if not isinstance(item, dict):
                    continue

                name = item.get("name", "Unknown item")
                quantity = item.get("quantity", 1)
                price = item.get("price")

                if price is not None:
                    lines.append(
                        f"- {name} x {quantity}: ${float(price):.2f}"
                    )
                else:
                    lines.append(
                        f"- {name} x {quantity}"
                    )

        total_price = order_summary.get("total_price")
        if total_price is not None:
            lines.append(
                f"Total price: ${float(total_price):.2f}"
            )

        delivery_time = order_summary.get("delivery_time_minutes")
        if delivery_time is not None:
            lines.append(
                f"Estimated delivery: {delivery_time} minutes"
            )

        if comparison_data:
            lines.extend([
                "",
                "Price comparison:",
            ])
            for platform, data in comparison_data.items():
                if not isinstance(data, dict):
                    continue
                total = data.get("total")
                if total is not None:
                    comparison_line = (
                        f"- {platform}: ${float(total):.2f}"
                    )
                else:
                    comparison_line = f"- {platform}"
                delivery = data.get("delivery_time_minutes")
                if delivery is not None:
                    comparison_line += (
                        f", {delivery} min"
                    )
                discount = data.get("discount", 0)
                if discount:
                    comparison_line += (
                        f", discount ${float(discount):.2f}"
                    )
                lines.append(comparison_line)

        return "\n".join(lines)

In [ ]:
import asyncio
from typing import Dict, Any, List, Optional

try:
    from .mcp_tools import search_items, get_prices, get_offers, compare_options, create_order, track_order, cancel_order
    from .memory_store import MemoryStore
    from .price_comparator import PriceComparator
    from .hitl_workflow import HITLWorkflow
except ImportError:
    from mcp_tools import search_items, get_prices, get_offers, compare_options, create_order, track_order, cancel_order
    from memory_store import MemoryStore
    from price_comparator import PriceComparator
    from hitl_workflow import HITLWorkflow


DEFAULT_PLATFORMS = ["zomato", "swiggy", "doordash", "deliveroo", "instacart"]


class WorkflowOrchestrator:
    """Orchestrates multi-step quick commerce workflow"""

    def __init__(
        self,
        memory_store: Optional[MemoryStore] = None,
        price_comparator: Optional[PriceComparator] = None,
        hitl_workflow: Optional[HITLWorkflow] = None
    ):
        """
        Initialize workflow orchestrator.

        Args:
            memory_store: Memory store instance
            price_comparator: Price comparator instance
            hitl_workflow: HITL workflow instance
        """
        self.memory_store = memory_store
        self.price_comparator = price_comparator or PriceComparator()
        self.hitl_workflow = hitl_workflow or HITLWorkflow()
        self.state: Dict[str, Dict[str, Any]] = {}

    async def execute_workflow(
        self,
        user_query: str,
        location: str,
        user_id: str,
        platforms: Optional[List[str]] = None,
        address: Optional[str] = None,
        payment_info: Optional[Dict[str, Any]] = None,
        workflow_id: Optional[str] = None,
    ) -> Dict[str, Any]:
        """
        Execute complete workflow: search -> compare -> approve -> order -> track.

        Args:
            user_query: User's search query
            location: Delivery location
            user_id: User identifier (required for authorization/HITL/memory)
            platforms: Optional list of platforms (None = all platforms)
            address: Delivery address, required before order creation
            payment_info: Payment information, required before order creation
            workflow_id: Optional identifier used to checkpoint/resume the workflow

        Returns:
            Workflow result with order details
        """
        if not isinstance(user_id, str) or not user_id.strip():
            return {"status": "failed", "error": "user_id is required"}

        platforms = platforms or DEFAULT_PLATFORMS
        workflow_id = workflow_id or f"wf_{user_id}_{int(asyncio.get_event_loop().time() * 1000)}"

        # Checkpoint the parameters so this workflow can be resumed if interrupted.
        self.state[workflow_id] = {
            "user_query": user_query,
            "location": location,
            "user_id": user_id,
            "platforms": platforms,
            "address": address,
            "payment_info": payment_info,
            "workflow_id": workflow_id,
        }

        completed_steps: List[str] = []
        step_context: Dict[str, Any] = {"user_id": user_id}

        try:
            # Step 0: Personalization from memory
            user_context = self._integrate_memory(user_id)

            # Step 1: Search Phase (parallel across platforms)
            search_results = await self._parallelize_platform_searches(
                user_query, location, user_id, platforms
            )
            completed_steps.append("search")

            if not search_results:
                return {"status": "failed", "error": "No search results from any platform"}

            # Step 2: Comparison Phase
            comparison = await self._integrate_comparison(search_results, user_id)
            completed_steps.append("comparison")

            best_option = comparison.get("best_option")
            preferred_platform = user_context.get("preferred_platform")

            if best_option and best_option.get("platform") in search_results:
                selected_platform = best_option["platform"]
            elif preferred_platform and preferred_platform in search_results:
                selected_platform = preferred_platform
            else:
                selected_platform = next(iter(search_results.keys()), None)

            if not selected_platform:
                return {"status": "failed", "error": "No platform available"}

            step_context["platform"] = selected_platform

            order_items = search_results[selected_platform].get("data", {}).get("items", [])

            order_summary = {
                "platform": selected_platform,
                "items": order_items,
                "total_price": (
                    best_option.get("final_price", best_option.get("total"))
                    if best_option else None
                ),
            }

            # Step 3: Approval Phase (Human-in-the-loop)
            approval_id = self.hitl_workflow.request_order_approval(
                order_summary, user_id, comparison.get("comparison")
            )

            approval_result = await self.hitl_workflow.wait_for_approval(approval_id)
            completed_steps.append("approval")

            if approval_result.get("status") != "approved":
                return {
                    "status": "rejected",
                    "approval": approval_result,
                    "platform": selected_platform,
                }

            if not address or not payment_info:
                return {
                    "status": "failed",
                    "error": "address and payment_info are required to place an order",
                    "platform": selected_platform,
                }

            # Step 4: Order Creation
            order_result = await self._execute_step(
                "create_order",
                create_order,
                selected_platform,
                order_items,
                address,
                user_id,
                payment_info,
            )

            if not isinstance(order_result, dict) or not order_result.get("success"):
                return await self._handle_step_failure(
                    "create_order",
                    RuntimeError(order_result.get("error") if isinstance(order_result, dict) else "Order creation failed"),
                    step_context,
                )

            completed_steps.append("create_order")

            order_id = order_result.get("data", {}).get("order_id")
            step_context["order_id"] = order_id

            if self.memory_store:
                self.memory_store.store_order_history({
                    "platform": selected_platform,
                    "items": order_items,
                    "order_id": order_id,
                })

            # Step 5: Tracking Phase
            tracking = None
            if order_id:
                tracking = await self._execute_step(
                    "track_order", track_order, selected_platform, user_id, order_id
                )
                completed_steps.append("track_order")

            self.state.pop(workflow_id, None)

            return {
                "status": "completed",
                "workflow_id": workflow_id,
                "order_id": order_id,
                "platform": selected_platform,
                "tracking": tracking,
            }

        except Exception as exc:
            await self._rollback_workflow(completed_steps, step_context)
            return await self._handle_step_failure("execute_workflow", exc, step_context)

    async def _execute_step(
        self,
        step_name: str,
        step_func,
        *args,
        **kwargs
    ) -> Any:
        """
        Execute individual workflow step, tolerating sync or async callables
        and converting failures into structured error results.
        """
        try:
            if asyncio.iscoroutinefunction(step_func):
                return await step_func(*args, **kwargs)
            return await asyncio.to_thread(step_func, *args, **kwargs)
        except Exception as exc:
            return await self._handle_step_failure(
                step_name, exc, {"args": args, "kwargs": kwargs}
            )

    async def _handle_step_failure(
        self,
        step_name: str,
        error: Exception,
        context: Dict[str, Any]
    ) -> Dict[str, Any]:
        """
        Recover from step failures by returning a structured error result.
        """
        return {
            "status": "failed",
            "step": step_name,
            "error": str(error),
            "context": context,
        }

    async def _rollback_workflow(
        self,
        completed_steps: List[str],
        context: Dict[str, Any]
    ) -> None:
        """
        Rollback completed steps on failure (e.g. cancel an already-created order).
        """
        if "create_order" in completed_steps:
            order_id = context.get("order_id")
            platform = context.get("platform")
            user_id = context.get("user_id")

            if order_id and platform and user_id:
                await self._execute_step(
                    "cancel_order", cancel_order, platform, user_id, order_id
                )

    async def _parallelize_platform_searches(
        self,
        query: str,
        location: str,
        user_id: str,
        platforms: List[str]
    ) -> Dict[str, Any]:
        """
        Execute platform searches in parallel, tolerating individual failures.
        """
        tasks = [
            self._execute_step(
                f"search_{platform}", search_items, query, location, user_id, [platform]
            )
            for platform in platforms
        ]

        results = await asyncio.gather(*tasks)

        search_results: Dict[str, Any] = {}
        for platform, result in zip(platforms, results):
            if isinstance(result, dict) and result.get("status") == "failed":
                continue
            if isinstance(result, dict) and platform in result:
                search_results[platform] = result[platform]

        return search_results

    async def _integrate_comparison(
        self, search_results: Dict[str, Any], user_id: str
    ) -> Dict[str, Any]:
        """
        Use price_comparator to compare prices/offers across platforms and
        determine the best option.
        """
        valid_platforms = []
        price_tasks = []
        offer_tasks = []

        for platform, data in search_results.items():
            item_id = data.get("data", {}).get("item_id")
            if not item_id:
                continue
            valid_platforms.append(platform)
            price_tasks.append(
                self._execute_step(f"price_{platform}", get_prices, item_id, user_id, [platform])
            )
            offer_tasks.append(
                self._execute_step(f"offers_{platform}", get_offers, user_id, [platform])
            )

        prices: Dict[str, Any] = {}
        offers: Dict[str, Any] = {}

        if price_tasks:
            price_results = await asyncio.gather(*price_tasks)
            for platform, result in zip(valid_platforms, price_results):
                if isinstance(result, dict) and platform in result:
                    prices[platform] = result[platform].get("data", {})

        if offer_tasks:
            offer_results = await asyncio.gather(*offer_tasks)
            for platform, result in zip(valid_platforms, offer_results):
                if isinstance(result, dict) and platform in result:
                    offers[platform] = result[platform].get("data", {})

        price_comparison = self.price_comparator.compare_prices(prices) if prices else {}
        offer_comparison = self.price_comparator.compare_offers(offers) if offers else {}

        combined: Dict[str, Any] = {}
        for platform in set(price_comparison) | set(offer_comparison):
            combined[platform] = {
                **price_comparison.get(platform, {}),
                **offer_comparison.get(platform, {}),
            }

        best_option = self.price_comparator.get_best_option(combined) if combined else None

        return {"comparison": combined, "best_option": best_option}

    def _integrate_memory(self, user_id: str) -> Dict[str, Any]:
        """
        Retrieve user preferences, order history and favorites from memory store.
        """
        if not self.memory_store:
            return {}

        return {
            "preferred_platform": self.memory_store.get_preferred_platform(),
            "preferences": self.memory_store.retrieve_preferences(),
            "order_history": self.memory_store.get_order_history(limit=5),
            "favorite_items": self.memory_store.get_favorite_items(),
        }

    async def _resume_workflow(self, workflow_id: str) -> Dict[str, Any]:
        """
        Resume an interrupted workflow from its last checkpointed parameters.
        """
        checkpoint = self.state.get(workflow_id)

        if not checkpoint:
            return {"status": "failed", "error": f"No checkpoint found for workflow {workflow_id}"}

        return await self.execute_workflow(**checkpoint)


In [ ]:
import asyncio
import re
from typing import Dict, Any, Optional, List

try:
    from .llm import get_llm_client
    from .memory_store import MemoryStore
    from .price_comparator import PriceComparator
    from .hitl_workflow import HITLWorkflow
    from .orchestrator import WorkflowOrchestrator, DEFAULT_PLATFORMS
    from .mcp_tools import search_items, get_prices, get_offers, compare_options, create_order, track_order, cancel_order
except ImportError:
    from llm import get_llm_client
    from memory_store import MemoryStore
    from price_comparator import PriceComparator
    from hitl_workflow import HITLWorkflow
    from orchestrator import WorkflowOrchestrator, DEFAULT_PLATFORMS
    from mcp_tools import search_items, get_prices, get_offers, compare_options, create_order, track_order, cancel_order


class QuickCommerceAgent:
    """Main quick commerce agent"""

    VALID_INTENTS = ("order", "compare", "track", "cancel")

    def __init__(self, user_id: str):
        """
        Initialize agent.

        Args:
            user_id: User identifier
        """
        self.user_id = user_id
        self.llm = get_llm_client()
        self.memory_store = MemoryStore(user_id=self.user_id)
        self.price_comparator = PriceComparator(user_preferences=self.memory_store.retrieve_preferences())

        self.hitl_workflow = HITLWorkflow()
        self.orchestrator = WorkflowOrchestrator(
            memory_store=self.memory_store,
            price_comparator=self.price_comparator,
            hitl_workflow=self.hitl_workflow
        )

        self.conversation_history: List[Dict[str, Any]] = []

    async def run(self, user_message: str, location: str) -> Dict[str, Any]:
        """
        Main agent loop.

        Args:
            user_message: User's message/request
            location: Delivery location

        Returns:
            Agent response
        """
        try:
            intent = self._understand_intent(user_message)
        except Exception as exc:
            result = {"status": "error", "message": f"Could not understand request: {exc}"}
            self._update_context(user_message, result)
            return result

        if intent == "order":
            result = await self._handle_order_intent(user_message, location)
        elif intent == "compare":
            result = await self._handle_compare_intent(user_message, location)
        elif intent == "track":
            result = await self._handle_track_intent(user_message)
        elif intent == "cancel":
            result = await self._handle_cancel_intent(user_message)
        else:
            result = {"status": "unknown_intent", "message": "I didn't understand your request"}

        result["message"] = self._format_response(result, {"intent": intent})

        self._update_context(user_message, result)

        return result

    async def _understand_intent(self, user_message: str) -> str:
        """
        Understand user intent from message.

        Args:
            user_message: User's message

        Returns:
            Intent (order, compare, track, cancel)
        """
        messages = [
            {
                "role": "system",
                "content": (
                    "Classify the user's message into exactly one of these intents: "
                    "order, compare, track, cancel. Respond with only the intent word."
                ),
            },
            {"role": "user", "content": user_message},
        ]

        try:
            response = self.llm.chat_completion(messages, max_tokens=10)
            intent = str(response.choices[0].message.content).strip().lower()
        except Exception as exc:
            raise Exception("LLM client not implemented or failed to respond") from exc

        if intent not in self.VALID_INTENTS:
            raise Exception(f"Unknown intent: {intent}")

        return intent

    async def _handle_order_intent(self, user_message: str, location: str) -> Dict[str, Any]:
        """
        Handle order intent.

        Args:
            user_message: User's message
            location: Delivery location

        Returns:
            Order result
        """
        preferences = self.memory_store.retrieve_preferences()
        address = preferences.get("default_address") or location
        payment_info = preferences.get("default_payment_info")

        if not payment_info:
            return {
                "status": "needs_info",
                "message": "I need your payment details before I can place this order.",
            }

        # The orchestrator handles search -> compare -> HITL approval -> order -> tracking.
        return await self.orchestrator.execute_workflow(
            user_query=user_message,
            location=location,
            user_id=self.user_id,
            platforms=None,
            address=address,
            payment_info=payment_info,
            workflow_id=None,
        )

    async def _handle_compare_intent(self, user_message: str, location: str) -> Dict[str, Any]:
        """
        Handle compare intent.

        Args:
            user_message: User's message
            location: Delivery location

        Returns:
            Comparison result
        """
        search_results = await self.orchestrator._parallelize_platform_searches(
            user_message, location, self.user_id, DEFAULT_PLATFORMS
        )

        if not search_results:
            return {"status": "failed", "error": "No search results from any platform"}

        comparison = await self.orchestrator._integrate_comparison(search_results, self.user_id)

        return {
            "status": "success",
            "comparison": comparison.get("comparison", {}),
            "best_option": comparison.get("best_option"),
        }

    async def _handle_track_intent(self, user_message: str) -> Dict[str, Any]:
        """
        Handle track intent.

        Args:
            user_message: User's message

        Returns:
            Tracking result
        """
        order_id, platform = self._find_order_reference(user_message)

        if not order_id or not platform:
            return {"status": "error", "message": "Could not determine which order to track"}

        tracking = await asyncio.to_thread(track_order, platform, self.user_id, order_id)

        return {"status": "success", "platform": platform, "order_id": order_id, "tracking": tracking}

    async def _handle_cancel_intent(self, user_message: str) -> Dict[str, Any]:
        """
        Handle cancel intent.

        Args:
            user_message: User's message

        Returns:
            Cancellation result
        """
        order_id, platform = self._find_order_reference(user_message)

        if not order_id or not platform:
            return {"status": "error", "message": "Could not determine which order to cancel"}

        cancellation = await asyncio.to_thread(cancel_order, platform, self.user_id, order_id)

        return {"status": "success", "platform": platform, "order_id": order_id, "cancellation": cancellation}

    def _find_order_reference(self, user_message: str) -> tuple:
        """
        Locate an order_id/platform pair referenced by the user message, falling
        back to the most recent order if none is explicitly mentioned.
        """
        order_history = self.memory_store.get_order_history()

        for order in order_history:
            order_id = order.get("order_id")
            if order_id and order_id in user_message:
                return order_id, order.get("platform")

        if order_history:
            most_recent = order_history[0]
            return most_recent.get("order_id"), most_recent.get("platform")

        return None, None

    def _select_tools(self, intent: str) -> List[str]:
        """
        Select appropriate tools based on user intent.

        Args:
            intent: User intent

        Returns:
            List of tool names to use
        """
        tool_map = {
            "order": ["search_items", "get_prices", "get_offers", "create_order"],
            "compare": ["search_items", "get_prices", "get_offers", "compare_options"],
            "track": ["track_order"],
            "cancel": ["cancel_order"],
        }

        return tool_map.get(intent, [])

    def _update_context(self, user_message: str, response: Dict[str, Any]) -> None:
        """
        Update conversation context and memory.

        Args:
            user_message: User's message
            response: Agent response
        """
        self.conversation_history.append({"role": "user", "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": response})

        if not isinstance(response, dict):
            return

        platform = response.get("platform")
        if response.get("status") == "completed" and platform:
            self.memory_store.store_preference("preferred_platform", platform)

    async def _handle_approval_required(self, order_summary: Dict[str, Any],
                                       comparison_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Integrate HITL workflows for order approval.

        Args:
            order_summary: Order summary
            comparison_data: Price comparison data

        Returns:
            Approval result
        """
        approval_id = self.hitl_workflow.request_order_approval(
            order_summary, self.user_id, comparison_data
        )

        return await self.hitl_workflow.wait_for_approval(approval_id)

    def _format_response(self, result: Dict[str, Any], context: Dict[str, Any]) -> str:
        """
        Format agent response with context.

        Args:
            result: Agent result
            context: Conversation context

        Returns:
            Formatted response text
        """
        if not isinstance(result, dict):
            return str(result)

        status = result.get("status")

        if status == "completed":
            return f"Your order on {result.get('platform')} was placed! Order ID: {result.get('order_id')}"

        if status == "rejected":
            return "The order was not approved."

        if status in ("failed", "error"):
            return f"Sorry, something went wrong: {result.get('error', result.get('message', 'unknown error'))}"

        if status == "needs_info":
            return result.get("message", "I need more information to continue.")

        if "comparison" in result:
            lines = ["Here's how platforms compare:"]
            for platform, data in result.get("comparison", {}).items():
                price = data.get("final_price", data.get("total"))
                line = f"- {platform}: ${price:.2f}" if price is not None else f"- {platform}"
                lines.append(line)

            best_option = result.get("best_option")
            if best_option:
                lines.append(f"Best option: {best_option.get('platform')}")

            return "\n".join(lines)

        if "tracking" in result:
            tracking = result.get("tracking", {}) or {}
            return f"Order {result.get('order_id')} on {result.get('platform')}: {tracking.get('data', tracking)}"

        if "cancellation" in result:
            return f"Order {result.get('order_id')} on {result.get('platform')} has been cancelled."

        return result.get("message", "Done.")

    def _suggest_from_memory(self, user_message: str) -> Optional[Dict[str, Any]]:
        """
        Suggest items from order history.

        Args:
            user_message: User's message

        Returns:
            Suggestion or None
        """
        order_history = self.memory_store.get_order_history(limit=1)

        if not order_history:
            return None

        last_order = order_history[0]

        return {
            "suggestion": "order_your_usual",
            "platform": last_order.get("platform"),
            "items": last_order.get("items", []),
        }
